In [ ]:
import segyio
import obspy as obs
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt

from scipy.signal import butter, filtfilt, medfilt, correlate

In [ ]:
FILEPATH = 'input/seismic.segy'

In [ ]:
def load_segy(path, neg_off=True):
    if neg_off:
        c = -1
    else:
        c = 1
    with segyio.open(path, ignore_geometry=True) as f:
        dt = segyio.tools.dt(f) / 1e6  # microseconds -> seconds
        data = segyio.tools.collect(f.trace[:])  # shape (n_traces, n_samples)
        headers = dict(
            offset=f.attributes(segyio.TraceField.offset)[:]*c,
            ffid=f.attributes(segyio.TraceField.FieldRecord)[:],
            cdp=f.attributes(segyio.TraceField.CDP)[:],
            src_x=f.attributes(segyio.TraceField.SourceX)[:],
            src_y=f.attributes(segyio.TraceField.SourceY)[:],
            rec_x=f.attributes(segyio.TraceField.GroupX)[:],
            rec_y=f.attributes(segyio.TraceField.GroupY)[:],
            src_depth=f.attributes(segyio.TraceField.SourceDepth)[:],
            rec_depth=f.attributes(segyio.TraceField.ReceiverGroupElevation)[:],
        )
    return data, headers, dt

In [ ]:
data, headers, dt = load_segy(FILEPATH)

In [ ]:
def qc_summary(data, headers, dt):
    """Print basic QC stats and flag dead traces."""
    n_traces, n_samples = data.shape
    print(f"Traces: {n_traces}, Samples/trace: {n_samples}, dt: {dt * 1000:.3f} ms")
    print(f"Offset range: {headers['offset'].min()} to {headers['offset'].max()} m")
    print(f"Unique FFIDs: {len(np.unique(headers['ffid']))}")
    trace_energy = np.sum(data.astype(np.float64) ** 2, axis=1)
    dead = np.where(trace_energy == 0)[0]
    print(f"Dead traces: {len(dead)}")
    return dead

In [ ]:
qc_summary(data, headers, dt)

In [ ]:
def remove_dead_bad_traces(data, energy_threshold_factor=0.05):
    """Zero out dead traces and traces with anomalously low/high energy."""
    trace_energy = np.sum(data.astype(np.float64) ** 2, axis=1)
    nonzero = trace_energy[trace_energy > 0]
    median_energy = np.median(nonzero) if len(nonzero) else 0
    bad = np.where(
        (trace_energy == 0) | (trace_energy < energy_threshold_factor * median_energy)
    )[0]
    data_clean = data.copy()
    data_clean[bad] = 0
    return data_clean, bad

In [ ]:
data_clean, bad = remove_dead_bad_traces(data)

In [ ]:
bad

In [ ]:
headers

In [ ]:
plot_shot_gather(data_clean, headers, dt, ffid=3)
plt.show()

# Single common-offset gather, nearest to 300 m
plot_common_offset_gather(data_clean, headers, dt, offset_value=3237)
plt.show()

# First vs last shot gather side by side
plot_first_and_last_shot(data_clean, headers, dt)
plt.show()

# Nearest vs farthest common-offset gather side by side
plot_first_and_last_common_offset(data_clean, headers, dt)
plt.show()

In [ ]:
from scipy.linalg import solve_toeplitz
DECON_OP_LEN_MS = 120      # lunghezza operatore decon (ms)
DECON_PRE_WHITE = 0.001    # pre-whitening
DECON_DESIGN_T0 = 0.30     # inizio finestra di design (s)
DECON_DESIGN_T1 = 2.50     # fine finestra di design (s)
# =============================================================================
# Wiener Spiking Deconvolution
# =============================================================================

OP_LEN = int(DECON_OP_LEN_MS / 1000 / dt)

i0_des = int(DECON_DESIGN_T0 / dt)
i1_des = min(int(DECON_DESIGN_T1 / dt), data.shape[1])
design_slice = slice(i0_des, i1_des)

t_axis = np.arange(data.shape[1]) * dt


def spiking_decon(trace, op_len, pre_whiten, design_slice):
    """
    Wiener spiking deconvolution of a single trace.
    """

    trace = np.asarray(trace, dtype=np.float64)

    if (not np.all(np.isfinite(trace))) or np.all(trace == 0):
        return trace.astype(np.float32)

    seg = trace[design_slice].copy()

    if len(seg) < op_len:
        return trace.astype(np.float32)

    if np.all(seg == 0):
        return trace.astype(np.float32)

    # remove DC
    seg -= seg.mean()

    # autocorrelation
    r = np.correlate(seg, seg, mode="full")
    r = r[len(seg)-1:len(seg)-1+op_len]

    if (r[0] <= 0) or (not np.isfinite(r[0])):
        return trace.astype(np.float32)

    # pre-whitening
    r[0] *= (1.0 + pre_whiten)

    rhs = np.zeros(op_len)
    rhs[0] = 1.0

    try:
        op = solve_toeplitz((r, r), rhs)
    except Exception:
        return trace.astype(np.float32)

    out = np.convolve(trace, op, mode="same")

    if not np.all(np.isfinite(out)):
        return trace.astype(np.float32)

    return out.astype(np.float32)


print(f"Operator length : {OP_LEN} samples ({DECON_OP_LEN_MS} ms)")
print(f"Design window   : {DECON_DESIGN_T0:.2f} - {DECON_DESIGN_T1:.2f} s")
print(f"Pre-whitening   : {DECON_PRE_WHITE}")

# =============================================================================
# Apply deconvolution
# =============================================================================

traces_dc = np.empty_like(data)

for i in range(data.shape[0]):
    traces_dc[i] = spiking_decon(
        data[i],
        OP_LEN,
        DECON_PRE_WHITE,
        design_slice,
    )

    if (i + 1) % max(1, data.shape[0] // 5) == 0:
        print(f"{i+1}/{data.shape[0]} traces processed")


# =============================================================================
# QC - First shot gather
# =============================================================================

shot1 = headers["ffid"] == headers["ffid"][0]

offsets = headers["offset"][shot1]
order = np.argsort(offsets)

# data is (n_traces, n_samples)
g_b = data[shot1][order].T
g_a = traces_dc[shot1][order].T

clip_b = np.percentile(np.abs(g_b), 99)
clip_a = np.percentile(np.abs(g_a), 99)

freqs = np.fft.rfftfreq(g_b.shape[0], dt)

spec_b = np.mean(np.abs(np.fft.rfft(g_b, axis=0)), axis=1)
spec_b /= spec_b.max()

spec_a = np.mean(np.abs(np.fft.rfft(g_a, axis=0)), axis=1)
spec_a /= spec_a.max()

fig = plt.figure(figsize=(13, 9))

gs = fig.add_gridspec(
    2,
    2,
    height_ratios=[2.2, 1],
)

ax0 = fig.add_subplot(gs[0, 0])
ax1 = fig.add_subplot(gs[0, 1], sharey=ax0)
ax2 = fig.add_subplot(gs[1, :])

extent = [
    offsets[order][0],
    offsets[order][-1],
    t_axis[-1],
    0,
]

ax0.imshow(
    g_b,
    aspect="auto",
    cmap="gray",
    vmin=-clip_b,
    vmax=clip_b,
    extent=extent,
)

ax0.set_title(f"Pre-decon (clip ±{clip_b:.2e})")
ax0.set_xlabel("Offset (m)")
ax0.set_ylabel("Time (s)")

ax1.imshow(
    g_a,
    aspect="auto",
    cmap="gray",
    vmin=-clip_a,
    vmax=clip_a,
    extent=extent,
)

ax1.set_title(f"Post-decon (clip ±{clip_a:.2e})")
ax1.set_xlabel("Offset (m)")

ax2.semilogy(freqs, spec_b, lw=1.5, label="Pre-decon")
ax2.semilogy(freqs, spec_a, lw=1.5, label="Post-decon")

ax2.set_xlim(0, 0.5 / dt)
ax2.set_xlabel("Frequency (Hz)")
ax2.set_ylabel("Normalized spectrum")
ax2.set_title("Average amplitude spectrum")
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
shotL = headers["ffid"] == headers["ffid"][-1]

offsets = headers["offset"][shotL]
order = np.argsort(offsets)

# data is (n_traces, n_samples)
g_b = data[shotL][order].T
g_a = traces_dc[shotL][order].T

clip_b = np.percentile(np.abs(g_b), 99)
clip_a = np.percentile(np.abs(g_a), 99)

freqs = np.fft.rfftfreq(g_b.shape[0], dt)

spec_b = np.mean(np.abs(np.fft.rfft(g_b, axis=0)), axis=1)
spec_b /= spec_b.max()

spec_a = np.mean(np.abs(np.fft.rfft(g_a, axis=0)), axis=1)
spec_a /= spec_a.max()

fig = plt.figure(figsize=(13, 9))

gs = fig.add_gridspec(
    2,
    2,
    height_ratios=[2.2, 1],
)

ax0 = fig.add_subplot(gs[0, 0])
ax1 = fig.add_subplot(gs[0, 1], sharey=ax0)
ax2 = fig.add_subplot(gs[1, :])

extent = [
    offsets[order][0],
    offsets[order][-1],
    t_axis[-1],
    0,
]

ax0.imshow(
    g_b,
    aspect="auto",
    cmap="gray",
    vmin=-clip_b,
    vmax=clip_b,
    extent=extent,
)

ax0.set_title(f"Pre-decon (clip ±{clip_b:.2e})")
ax0.set_xlabel("Offset (m)")
ax0.set_ylabel("Time (s)")

ax1.imshow(
    g_a,
    aspect="auto",
    cmap="gray",
    vmin=-clip_a,
    vmax=clip_a,
    extent=extent,
)

ax1.set_title(f"Post-decon (clip ±{clip_a:.2e})")
ax1.set_xlabel("Offset (m)")

ax2.semilogy(freqs, spec_b, lw=1.5, label="Pre-decon")
ax2.semilogy(freqs, spec_a, lw=1.5, label="Post-decon")

ax2.set_xlim(0, 0.5 / dt)
ax2.set_xlabel("Frequency (Hz)")
ax2.set_ylabel("Normalized spectrum")
ax2.set_title("Average amplitude spectrum")
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
def bandpass_filter(data, dt, low_hz=3, high_hz=80, order=4):
    """Zero-phase Butterworth bandpass to cut low-freq swell noise and high-freq junk."""
    fs = 1.0 / dt
    nyq = 0.5 * fs
    b, a = butter(order, [low_hz / nyq, high_hz / nyq], btype="band")
    return filtfilt(b, a, data, axis=1)


def despike(data, kernel_size=5):
    """Median filter along time axis to knock out isolated spikes."""
    return medfilt(data, kernel_size=(1, kernel_size))

In [ ]:
data_filtered = bandpass_filter(traces_dc, dt)
data_filtered = despike(data_filtered)

In [ ]:
plot_shot_gather(data_filtered, headers, dt, ffid=3)
plt.show()

# Single common-offset gather, nearest to 300 m
plot_common_offset_gather(data_filtered, headers, dt, offset_value=3237)
plt.show()

# First vs last shot gather side by side
plot_first_and_last_shot(data_filtered, headers, dt)
plt.show()

# Nearest vs farthest common-offset gather side by side
plot_first_and_last_common_offset(data_filtered, headers, dt)
plt.show()

In [ ]:
def spherical_divergence_correction(data, dt, power=2.0, v_rms=None):
    """
    Compensate for geometric spreading loss.
    If v_rms (array over time, same length as n_samples) is given, uses
    t * v_rms^2 gain (more physically correct); otherwise falls back to t^power.
    """
    n_traces, n_samples = data.shape
    t = np.arange(n_samples) * dt
    t[0] = dt  # avoid zero-division at t=0
    gain = (v_rms ** 2) * t if v_rms is not None else t ** power
    return data * gain[np.newaxis, :]


def agc(data, dt, window_ms=500):
    """Automatic Gain Control using an RMS sliding window, per trace."""
    n_traces, n_samples = data.shape
    win = max(int(round((window_ms / 1000.0) / dt)), 1)
    kernel = np.ones(win) / win
    out = np.zeros_like(data, dtype=np.float64)
    for i in range(n_traces):
        trace = data[i].astype(np.float64)
        power = np.convolve(trace ** 2, kernel, mode="same")
        rms = np.sqrt(power + 1e-12)
        out[i] = trace / rms
    return out

In [ ]:
data_spheric = spherical_divergence_correction(data_filtered, dt)
data_acg = agc(data_spheric, dt)

In [ ]:
plot_shot_gather(data_spheric, headers, dt, ffid=3)
plt.show()

# Single common-offset gather, nearest to 300 m
plot_common_offset_gather(data_spheric, headers, dt, offset_value=3237)
plt.show()

# First vs last shot gather side by side
plot_first_and_last_shot(data_spheric, headers, dt)
plt.show()

# Nearest vs farthest common-offset gather side by side
plot_first_and_last_common_offset(data_spheric, headers, dt)
plt.show()

In [ ]:
plot_shot_gather(data_acg, headers, dt, ffid=3)
plt.show()

# Single common-offset gather, nearest to 300 m
plot_common_offset_gather(data_acg, headers, dt, offset_value=3237)
plt.show()

# First vs last shot gather side by side
plot_first_and_last_shot(data_acg, headers, dt)
plt.show()

# Nearest vs farthest common-offset gather side by side
plot_first_and_last_common_offset(data_acg, headers, dt)
plt.show()

In [ ]:
def water_column_static(data, dt, src_depth, rec_depth, v_water=1500.0):
    """
    Shift each trace to compensate for source/streamer depth so all traces
    are referenced to a common datum (e.g. sea surface). Marine equivalent
    of a land elevation static.
    src_depth, rec_depth: arrays (n_traces,) in metres, positive down.
    """
    n_traces, n_samples = data.shape
    shifted = np.zeros_like(data)
    for i in range(n_traces):
        t_shift = (src_depth[i] + rec_depth[i]) / v_water
        n_shift = int(round(t_shift / dt))
        if n_shift > 0:
            shifted[i, n_shift:] = data[i, : n_samples - n_shift]
        elif n_shift < 0:
            shifted[i, :n_shift] = data[i, -n_shift:]
        else:
            shifted[i] = data[i]
    return shifted

In [ ]:
data_corr = water_column_static(data_spheric, dt, headers["src_depth"], headers["rec_depth"])

In [ ]:
plot_shot_gather(data_corr, headers, dt, ffid=3)
plt.show()

# Single common-offset gather, nearest to 300 m
plot_common_offset_gather(data_corr, headers, dt, offset_value=3237)
plt.show()

# First vs last shot gather side by side
plot_first_and_last_shot(data_corr, headers, dt)
plt.show()

# Nearest vs farthest common-offset gather side by side
plot_first_and_last_common_offset(data_corr, headers, dt)
plt.show()

In [ ]:
def semblance_panel(cmp_gather, offsets, dt, v_range, t_step=4):
    """
    Compute a semblance (coherence) panel for one CMP gather.
    cmp_gather: (n_traces, n_samples)
    offsets:    (n_traces,) in metres
    v_range:    1D array of trial NMO velocities (m/s)
    t_step:     subsample every t_step samples on the time axis (for speed)

    Returns semblance[n_t0, n_vel], t0_axis, v_range
    Pick the velocity with max semblance at each t0 to build your velocity function.
    """
    n_traces, n_samples = cmp_gather.shape
    t0_axis = np.arange(0, n_samples, t_step) * dt
    semblance = np.zeros((len(t0_axis), len(v_range)))

    for vi, v in enumerate(v_range):
        for ti, t0 in enumerate(t0_axis):
            t_x = np.sqrt(t0 ** 2 + (offsets / v) ** 2)
            idx = np.round(t_x / dt).astype(int)
            valid = (idx >= 0) & (idx < n_samples)
            if valid.sum() < 2:
                continue
            amps = cmp_gather[valid, idx[valid]]
            num = np.sum(amps) ** 2
            den = valid.sum() * np.sum(amps ** 2) + 1e-9
            semblance[ti, vi] = num / den

    return semblance, t0_axis, v_range



In [ ]:

class InteractiveVelocityPicker:
    """
    Interactive velocity picker for Jupyter + ipympl (%matplotlib widget).
 
    Unlike pick_velocity_interactive() (which uses the blocking plt.ginput
    and can render as a blank/frozen canvas under ipympl -- ginput's blocking
    event loop conflicts with ipympl's async browser-based rendering), this
    uses live matplotlib event callbacks, which work correctly with ipympl.
 
    Usage (each block in its own Jupyter cell):
 
        %matplotlib widget
        picker = InteractiveVelocityPicker(semblance, t0_axis, v_range)
        picker.show()
 
        # ^ click directly on the figure that appears above:
        #   left-click  -> add a pick at (velocity, time)
        #   right-click -> remove the most recent pick
        # the red line updates live as you click.
 
        # once you're done picking, in a later cell:
        times_picked, velocities_picked = picker.get_picks()
        velocity_t0 = picker.get_velocity_function(full_t0)
 
    If you don't see any figure at all, confirm `ipympl` is installed
    (`pip install ipympl --break-system-packages`) and that you ran
    `%matplotlib widget` in a cell BEFORE creating the picker.
    """
 
    def __init__(self, semblance, t0_axis, v_range, cmap="viridis",
                 figsize=(7, 8), pctl=99):
        self.t0_axis = t0_axis
        self.v_range = v_range
        self.times = []
        self.velocities = []
 
        self.fig, self.ax = plt.subplots(figsize=figsize)
        self.ax.imshow(
            semblance,
            cmap=cmap,
            aspect="auto",
            vmin=0,
            vmax=np.nanpercentile(semblance, pctl),
            extent=[v_range[0], v_range[-1], t0_axis[-1], 0],
        )
        self.ax.set_xlabel("Velocity [m/s]")
        self.ax.set_ylabel("Time [s]")
        self.ax.set_title("Left-click: add pick | Right-click: remove last")
        (self.line,) = self.ax.plot([], [], "r.-", markersize=8, linewidth=1.5)
 
        self.fig.canvas.mpl_connect("button_press_event", self._on_click)
 
    def show(self):
        """Explicitly (re-)display the figure -- handy if it didn't render
        automatically on creation."""
        return self.fig
 
    def _on_click(self, event):
        if event.inaxes != self.ax or event.xdata is None or event.ydata is None:
            return
        if event.button == 1:      # left click: add
            self.velocities.append(event.xdata)
            self.times.append(event.ydata)
        elif event.button == 3:    # right click: remove last
            if self.times:
                self.times.pop()
                self.velocities.pop()
        self._redraw()
 
    def _redraw(self):
        if self.times:
            order = np.argsort(self.times)
            t_sorted = np.array(self.times)[order]
            v_sorted = np.array(self.velocities)[order]
            self.line.set_data(v_sorted, t_sorted)
        else:
            self.line.set_data([], [])
        self.fig.canvas.draw_idle()
 
    def get_picks(self):
        """Returns (times_picked, velocities_picked), sorted by time."""
        if not self.times:
            return np.array([]), np.array([])
        order = np.argsort(self.times)
        t_sorted = np.array(self.times)[order]
        v_sorted = np.array(self.velocities)[order]
        return t_sorted, v_sorted
 
    def get_velocity_function(self, full_t0):
        """Interpolate current picks onto a full-sample-rate time axis."""
        t_sorted, v_sorted = self.get_picks()
        if len(t_sorted) < 2:
            raise ValueError("Need at least 2 picks to build a velocity function")
        return np.interp(full_t0, t_sorted, v_sorted)

In [ ]:
list_enough_cdp = []
for cdp in np.unique(headers["cdp"]):
    if (headers["cdp"] == cdp).sum() >= 60:
        #print(np.where(headers["cdp"] == cdp)[0].sum())
        list_enough_cdp.append(cdp)

In [ ]:
list_enough_cdp = np.random.choice(list_enough_cdp, 10)

In [ ]:
full_t0 = np.arange(data.shape[1]) * dt
v_range = np.arange(1400, 4500, 20)
STORE = {}
control_picks = {}
pickers = {}
semblances = {}

# --- Cell 1: build a picker for each control CMP (run once) ---
for cdp in list_enough_cdp:
    cdp = int(cdp)
    cmp_mask = headers["cdp"] == cdp
    cmp_gather = data_corr[cmp_mask]
    cmp_offsets = headers["offset"][cmp_mask]
    semblance, t0_axis, v_range = semblance_panel(cmp_gather, cmp_offsets, dt, v_range)
    semblances[cdp] = (semblance, t0_axis)

In [ ]:
semblances.keys()

In [ ]:
KEY = 1137

STORE[KEY] = dict()

In [ ]:
%matplotlib widget
picker = InteractiveVelocityPicker(semblances[KEY][0], semblances[KEY][1], v_range)
picker.show()

In [ ]:
times_picked, velocities_picked = picker.get_picks()
velocity_t0 = picker.get_velocity_function(full_t0)

In [ ]:
STORE[KEY]['times'] = times_picked
STORE[KEY]['picked_v'] = velocities_picked
STORE[KEY]['vel_int'] = velocity_t0

In [ ]:
STORE.keys()

In [ ]:
velocity_dict = {
    key: value["vel_int"]
    for key, value in STORE.items()
}

In [ ]:
all_cdps = np.unique(headers["cdp"])
velocity_field = build_velocity_field(velocity_dict, all_cdps, full_t0)

In [ ]:
velocity_field.shape

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(
    velocity_field.T,
    aspect="auto",
    cmap="jet",
    extent=[all_cdps[0], all_cdps[-1], full_t0[-1], 0],
)
ax.set_xlabel("CDP")
ax.set_ylabel("Time [s]")
ax.set_title("Velocity Field")
plt.colorbar(im, ax=ax, label="Velocity [m/s]")

In [ ]:
stacked_section, fold_section = nmo_stack_full_line(
    data_corr, headers, dt, all_cdps, velocity_field, stretch_mute=0.3
)

In [ ]:
def nmo_correction(cmp_gather, offsets, dt, velocity_t0, stretch_mute=0.3):
    """
    Apply NMO using a velocity function velocity_t0(t0) (same length as n_samples,
    interpolate your picked semblance velocities onto the full time axis first).
    stretch_mute: fractional stretch beyond which samples are muted (typ. 0.2-0.5).
    """
    n_traces, n_samples = cmp_gather.shape
    t0 = np.arange(n_samples) * dt
    nmo_data = np.zeros_like(cmp_gather, dtype=np.float64)

    for i, x in enumerate(offsets):
        t_x = np.sqrt(t0 ** 2 + (x / velocity_t0) ** 2)
        with np.errstate(divide="ignore", invalid="ignore"):
            stretch = np.where(t0 > 0, (t_x - t0) / np.where(t0 > 0, t0, 1), 0)
        amp = np.interp(t_x, t0, cmp_gather[i], left=0, right=0)
        amp[stretch > stretch_mute] = 0
        nmo_data[i] = amp

    return nmo_data

In [ ]:
nmo_gather = nmo_correction(cmp_gather, cmp_offsets, dt, velocity_t0, stretch_mute=0.3)

In [ ]:
plot_cmp_gather(cmp_gather, nmo_gather, cmp_offsets, dt, title=f"CDP {list_enough_cdp[0]}")

In [ ]:
# --- QC: compare one CDP before/after NMO ---
cdp_to_check = list_enough_cdp[0]

cmp_mask = headers["cdp"] == cdp_to_check
cmp_gather = data_corr[cmp_mask]
cmp_offsets = headers["offset"][cmp_mask]

row = np.where(all_cdps == cdp_to_check)[0][0]
velocity_t0 = velocity_field[row]

nmo_gather = nmo_correction(cmp_gather, cmp_offsets, dt, velocity_t0, stretch_mute=0.3)

plot_cmp_gather(cmp_gather, nmo_gather, cmp_offsets, dt, title=f"CDP {cdp_to_check}")
plt.show()

In [ ]:
def stack_cmp(nmo_gather):
    """Fold-normalized stack of an NMO-corrected CMP gather."""
    fold = np.sum(nmo_gather != 0, axis=0)
    fold_safe = np.where(fold == 0, 1, fold)
    return np.sum(nmo_gather, axis=0) / fold_safe
 
 
def build_velocity_field(control_picks, all_cdps, full_t0):
    """
    Build a spatially-continuous velocity field (all_cdps x n_samples) from
    velocity functions picked at a sparse set of control CMPs.
 
    control_picks: dict {cdp: velocity_t0} where each velocity_t0 is already
    the full-sample-rate array (length == len(full_t0)) for that control CMP
    -- i.e. what you get from np.interp(full_t0, t0_axis_or_times_picked,
    v_picked) for each control CMP's velocity analysis.
 
    all_cdps: 1D array of every CMP/CDP number in the line (e.g.
    np.unique(headers["cdp"])), sorted or not -- output rows follow this order.
 
    For each time sample, linearly interpolates across CDP number between
    control points (and holds flat beyond the first/last control CDP -- same
    edge behavior as np.interp). This assumes velocity varies smoothly along
    the line between your control points, which is standard practice; pick
    more control CMPs in zones with rapid lateral velocity change.
 
    Returns velocity_field: array (len(all_cdps), len(full_t0)).
    """
    control_cdps = np.array(sorted(control_picks.keys()))
    V_control = np.array([control_picks[cdp] for cdp in control_cdps])  # (n_control, n_samples)
 
    all_cdps = np.asarray(all_cdps)
    n_samples = len(full_t0)
    velocity_field = np.zeros((len(all_cdps), n_samples))
 
    for j in range(n_samples):
        velocity_field[:, j] = np.interp(all_cdps, control_cdps, V_control[:, j])
 
    return velocity_field
 
 
def nmo_stack_full_line(data, headers, dt, all_cdps, velocity_field,
                         stretch_mute=0.3, verbose=True):
    """
    Apply NMO (using each CMP's own row from velocity_field) and stack, for
    every CDP in all_cdps. This is the loop that turns your per-control-CMP
    velocity picks into a full stacked section.
 
    velocity_field: array (len(all_cdps), n_samples) -- e.g. from
    build_velocity_field(). Row i is the velocity_t0 function used for
    all_cdps[i].
 
    Returns:
        stacked_section: array (len(all_cdps), n_samples)
        fold_section:     array (len(all_cdps), n_samples) -- trace count
                          contributing to each stacked sample (useful for QC:
                          low-fold zones are less reliable).
    """
    n_samples = data.shape[1]
    stacked_section = np.zeros((len(all_cdps), n_samples), dtype=np.float32)
    fold_section = np.zeros((len(all_cdps), n_samples), dtype=np.int32)
 
    for i, cdp in enumerate(all_cdps):
        cmp_mask = headers["cdp"] == cdp
        if cmp_mask.sum() == 0:
            continue
 
        cmp_gather = data[cmp_mask]
        cmp_offsets = headers["offset"][cmp_mask]
        velocity_t0 = velocity_field[i]
 
        nmo_gather = nmo_correction(cmp_gather, cmp_offsets, dt, velocity_t0,
                                     stretch_mute=stretch_mute)
        stacked_section[i] = stack_cmp(nmo_gather)
        fold_section[i] = np.sum(nmo_gather != 0, axis=0)
 
        if verbose and (i + 1) % max(1, len(all_cdps) // 10) == 0:
            print(f"  stacked {i + 1}/{len(all_cdps)} CMPs")
 
    return stacked_section, fold_section
 



In [ ]:




# ---------------------------------------------------------------------------
# 7. STACK
# ---------------------------------------------------------------------------

def stack_cmp(nmo_gather):
    """Fold-normalized stack of an NMO-corrected CMP gather."""
    fold = np.sum(nmo_gather != 0, axis=0)
    fold_safe = np.where(fold == 0, 1, fold)
    return np.sum(nmo_gather, axis=0) / fold_safe


# ---------------------------------------------------------------------------
# 8. RESIDUAL STATIC CORRECTION
# ---------------------------------------------------------------------------

def residual_statics_xcorr(cmp_gather, pilot_trace, dt, max_shift_ms=20):
    """
    Crosscorrelation-based residual statics: find the time shift per trace
    that best aligns it with a pilot trace (e.g. the current stack, or a
    nearby high-fold stacked trace). Run iteratively with re-stacking for
    surface-consistent convergence.
    Returns integer sample shifts (n_traces,).
    """
    max_shift = int(round((max_shift_ms / 1000.0) / dt))
    shifts = np.zeros(cmp_gather.shape[0], dtype=int)

    for i, trace in enumerate(cmp_gather):
        xcorr = correlate(trace, pilot_trace, mode="full")
        lags = np.arange(-(len(pilot_trace) - 1), len(trace))
        center = len(xcorr) // 2
        lo, hi = center - max_shift, center + max_shift + 1
        lo, hi = max(lo, 0), min(hi, len(xcorr))
        best_lag = lags[lo:hi][np.argmax(xcorr[lo:hi])]
        shifts[i] = best_lag

    return shifts


def apply_static_shifts(data, shifts):
    """Apply integer-sample static shifts to each trace."""
    shifted = np.zeros_like(data)
    for i, s in enumerate(shifts):
        if s > 0:
            shifted[i, s:] = data[i, :-s]
        elif s < 0:
            shifted[i, :s] = data[i, -s:]
        else:
            shifted[i] = data[i]
    return shifted


# ---------------------------------------------------------------------------
# EXAMPLE END-TO-END USAGE (adapt to your CDP sorting / velocity picks)
# ---------------------------------------------------------------------------


In [ ]:

if __name__ == "__main__":
    path = "your_file.sgy"

    # 1. Load & QC
    data, headers, dt = load_segy(path)
    dead = qc_summary(data, headers, dt)

    # 2. Noise attenuation
    data, bad = remove_dead_bad_traces(data)
    data = bandpass_filter(data, dt, low_hz=3, high_hz=80)
    data = despike(data, kernel_size=5)

    # 3. Amplitude correction
    data = spherical_divergence_correction(data, dt, power=2.0)
    # optionally also: data = agc(data, dt, window_ms=500)

    # 4. Datum / water-column static correction
    data = water_column_static(data, dt, headers["src_depth"], headers["rec_depth"])

    # --- from here on, work per-CMP-gather ---
    # example for a single CMP (you'll loop this over all CMPs):
    cmp_mask = headers["cdp"] == headers["cdp"][0]
    cmp_gather = data[cmp_mask]
    cmp_offsets = headers["offset"][cmp_mask]

    # 5. Velocity analysis
    v_range = np.arange(1400, 4500, 20)
    semblance, t0_axis, v_range = semblance_panel(cmp_gather, cmp_offsets, dt, v_range)
    v_picked_coarse = pick_velocity_from_semblance(semblance, t0_axis, v_range)
    n_samples = cmp_gather.shape[1]
    full_t0 = np.arange(n_samples) * dt
    velocity_t0 = np.interp(full_t0, t0_axis, v_picked_coarse)

    # 6. NMO correction
    nmo_gather = nmo_correction(cmp_gather, cmp_offsets, dt, velocity_t0, stretch_mute=0.3)

    # 7. Stack
    stacked_trace = stack_cmp(nmo_gather)

    # 8. Residual statics (example: align raw CMP traces to the stack, then re-stack)
    shifts = residual_statics_xcorr(nmo_gather, stacked_trace, dt, max_shift_ms=20)
    nmo_gather_corrected = apply_static_shifts(nmo_gather, shifts)
    stacked_trace_final = stack_cmp(nmo_gather_corrected)

In [ ]:
"""
Plotting helpers for QC of shot gathers and common-offset gathers.

Assumes you've already loaded data with load_segy() from
seismic_processing_pipeline.py, so you have:
    data:    (n_traces, n_samples) numpy array
    headers: dict with 'offset', 'ffid', 'cdp', etc. (n_traces,) arrays
    dt:      sample interval in seconds

Requires: numpy, matplotlib
"""

import numpy as np
import matplotlib.pyplot as plt


def _clip_val(gather, pctl=98):
    """Symmetric clip value for gray-scale display, based on amplitude percentile."""
    return np.percentile(np.abs(gather), pctl) or 1.0


def plot_shot_gather(data, headers, dt, ffid, ax=None, pctl=98, title=None):
    """
    Plot all traces belonging to a single FFID (shot gather), offset on x-axis,
    time on y-axis (downward), matching a standard 'Shot gather FFID=n' display.
    """
    mask = headers["ffid"] == ffid
    if mask.sum() == 0:
        raise ValueError(f"No traces found for FFID={ffid}")

    gather = data[mask]
    offsets = headers["offset"][mask]

    order = np.argsort(offsets)
    gather = gather[order]
    offsets = offsets[order]

    n_samples = gather.shape[1]
    t_max = n_samples * dt

    clip = _clip_val(gather, pctl)

    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 8))
    ax.imshow(
        gather.T,
        cmap="gray",
        aspect="auto",
        vmin=-clip,
        vmax=clip,
        extent=[offsets.min(), offsets.max(), t_max, 0],
    )
    ax.set_xlabel("Offset [m]")
    ax.set_ylabel("Time [s]")
    ax.set_title(title or f"Shot gather FFID={ffid}")
    return ax


def plot_common_offset_gather(data, headers, dt, offset_value, tolerance=1.0,
                               ax=None, pctl=98, title=None, sort_by="cdp"):
    """
    Plot all traces sharing (approximately) the same offset -- a common-offset
    section. offset_value: target offset in metres. tolerance: matching window (m).
    sort_by: header key to order traces along x-axis (e.g. 'cdp', or None to keep file order).
    """
    mask = np.where(headers["offset"] == offset_value)[0]
    if mask.sum() == 0:
        raise ValueError(f"No traces found near offset={offset_value} (tol={tolerance})")

    gather = data[mask]

    if sort_by is not None:
        order = np.argsort(headers[sort_by][mask])
        gather = gather[order]

    n_samples = gather.shape[1]
    t_max = n_samples * dt

    clip = _clip_val(gather, pctl)

    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 8))
    ax.imshow(
        gather.T,
        cmap="gray",
        aspect="auto",
        vmin=-clip,
        vmax=clip,
        extent=[0, gather.shape[0], t_max, 0],
    )
    ax.set_xlabel("Trace index")
    ax.set_ylabel("Time [s]")
    ax.set_title(title or f"Common offset gather (offset≈{offset_value} m)")
    return ax


def plot_first_and_last_shot(data, headers, dt, pctl=98):
    """Convenience: side-by-side plot of the first and last FFID shot gathers."""
    ffids = np.unique(headers["ffid"])
    first_ffid, last_ffid = ffids.min(), ffids.max()

    fig, axes = plt.subplots(1, 2, figsize=(14, 8))
    plot_shot_gather(data, headers, dt, first_ffid, ax=axes[0], pctl=pctl,
                      title=f"First shot gather (FFID={first_ffid})")
    plot_shot_gather(data, headers, dt, last_ffid, ax=axes[1], pctl=pctl,
                      title=f"Last shot gather (FFID={last_ffid})")
    plt.tight_layout()
    return fig, axes


def plot_first_and_last_common_offset(data, headers, dt, pctl=98, sort_by="cdp"):
    """Convenience: side-by-side plot of the nearest and farthest offset gathers."""
    min_offset = headers["offset"].min()
    max_offset = headers["offset"].max()

    fig, axes = plt.subplots(1, 2, figsize=(14, 8))
    plot_common_offset_gather(data, headers, dt, min_offset, ax=axes[0], pctl=pctl,
                               sort_by=sort_by, title=f"Nearest common offset (~{min_offset} m)")
    plot_common_offset_gather(data, headers, dt, max_offset, ax=axes[1], pctl=pctl,
                               sort_by=sort_by, title=f"Farthest common offset (~{max_offset} m)")
    plt.tight_layout()
    return fig, axes




In [ ]:
def plot_decon_qc(data_pre, data_post, headers, dt, ffid, pctl=99):
    """
    QC plot for spiking deconvolution: pre/post shot gather (independently
    clipped) plus mean amplitude spectrum comparison for that shot.
    data_pre, data_post: full (n_traces, n_samples) arrays, same headers.
    """
    mask = headers["ffid"] == ffid
    if mask.sum() == 0:
        raise ValueError(f"No traces found for FFID={ffid}")
 
    offsets = headers["offset"][mask]
    order = np.argsort(offsets)
 
    g_b = data_pre[mask][order].T   # (n_samples, n_traces_in_shot)
    g_a = data_post[mask][order].T
    off_sorted = offsets[order]
 
    n_samples = g_b.shape[0]
    t_axis = np.arange(n_samples) * dt
 
    clip_b = np.percentile(np.abs(g_b), pctl) + 1e-12
    clip_a = np.percentile(np.abs(g_a), pctl) + 1e-12
 
    freqs = np.fft.rfftfreq(n_samples, d=dt)
    spec_b = np.mean(np.abs(np.fft.rfft(g_b, axis=0)), axis=1)
    spec_b = spec_b / spec_b.max()
    spec_a = np.mean(np.abs(np.fft.rfft(g_a, axis=0)), axis=1)
    spec_a = spec_a / spec_a.max()
 
    fig = plt.figure(figsize=(13, 9))
    gs = fig.add_gridspec(2, 2, height_ratios=[2.2, 1])
    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[0, 1], sharey=ax0)
    ax2 = fig.add_subplot(gs[1, :])
 
    ext = [off_sorted[0], off_sorted[-1], t_axis[-1], 0]
    ax0.imshow(g_b, aspect="auto", cmap="gray", vmin=-clip_b, vmax=clip_b, extent=ext)
    ax0.set_title(f"Pre-decon  (clip ±{clip_b:.2e})")
    ax0.set_xlabel("Offset [m]")
    ax0.set_ylabel("Time [s]")
 
    ax1.imshow(g_a, aspect="auto", cmap="gray", vmin=-clip_a, vmax=clip_a, extent=ext)
    ax1.set_title(f"Post-decon (clip ±{clip_a:.2e})")
    ax1.set_xlabel("Offset [m]")
 
    ax2.semilogy(freqs, spec_b, label="Pre-decon", lw=1.2)
    ax2.semilogy(freqs, spec_a, label="Post-decon", lw=1.2)
    ax2.set_xlim(0, 0.5 / dt)
    ax2.set_xlabel("Frequency [Hz]")
    ax2.set_ylabel("Normalized spectrum (log)")
    ax2.set_title(f"Mean spectrum, shot FFID={ffid} — spectral whitening check")
    ax2.legend()
    ax2.grid(alpha=0.3)
 
    plt.tight_layout()
    return fig, (ax0, ax1, ax2)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_cmp_gather(cmp_gather, nmo_gather, cmp_offsets, dt, title="CMP Gather vs NMO",
                     clip_pct=99, cmap="gray"):

    n_traces, n_samples = cmp_gather.shape
    t_axis = np.arange(n_samples) * dt

    # sort traces by offset so the plot reads left-to-right correctly
    order = np.argsort(cmp_offsets)
    gather_sorted = cmp_gather[order]
    offsets_sorted = cmp_offsets[order]

    nmo_sorted = nmo_gather[order]
  

    # symmetric clip for balanced color scale
    vmax = np.percentile(np.abs(gather_sorted), clip_pct)
    vmin = -vmax

    fig, ax = plt.subplots(1, 2, figsize=(14, 6))

    # First plot
    im1 = ax[0].imshow(
        gather_sorted.T,
        aspect="auto",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        extent=[
            offsets_sorted.min(),
            offsets_sorted.max(),
            t_axis[-1],
            t_axis[0],
        ],
    )

    ax[0].set_xlabel("Offset (m)")
    ax[0].set_ylabel("Time (s)")
    ax[0].set_title("Gather")

    # Second plot
    im2 = ax[1].imshow(
        nmo_sorted.T,      # <-- replace with your second gather
        aspect="auto",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        extent=[
            offsets_sorted.min(),
            offsets_sorted.max(),
            t_axis[-1],
            t_axis[0],
        ],
    )

    ax[1].set_xlabel("Offset (m)")
    ax[1].set_ylabel("Time (s)")
    ax[1].set_title("NMO Gather")

    plt.tight_layout()
    plt.show()